In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('car_data.csv')

current_year = 2026 
df['Car_Age'] = current_year - df['Year']

df_encoded = pd.get_dummies(df, drop_first=True)

X = df_encoded.drop(['Selling_Price', 'Car_Name', 'Year'], axis=1, errors='ignore')
y = df_encoded['Selling_Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

def predict_new_car(year, present_price, kms_driven, owner, fuel_type, seller_type, transmission):
    input_data = pd.DataFrame({
        'Present_Price': [present_price],
        'Kms_Driven': [kms_driven],
        'Owner': [owner],
        'Car_Age': [2026 - year]
    })
    
    for col in X.columns:
        if col not in input_data.columns:
            input_data[col] = 0
            
    if fuel_type == 'Diesel':
        input_data['Fuel_Type_Diesel'] = 1
    elif fuel_type == 'Petrol':
        input_data['Fuel_Type_Petrol'] = 1
        
    if seller_type == 'Individual':
        input_data['Seller_Type_Individual'] = 1
        
    if transmission == 'Manual':
        input_data['Transmission_Manual'] = 1
        
    input_data = input_data[X.columns]
    input_scaled = scaler.transform(input_data)
    prediction = rf_model.predict(input_scaled)
    return np.round(prediction[0], 2)

print("--- Enter Vehicle Details for Prediction ---")
input_year = int(input("Enter the manufacturing year (e.g., 2017): "))
input_present_price = float(input("Enter the present price in Lakhs (e.g., 9.85): "))
input_kms_driven = int(input("Enter kilometers driven (e.g., 6900): "))
input_owner = int(input("Enter number of previous owners (0, 1, 2...): "))
input_fuel_type = input("Enter Fuel Type (Petrol/Diesel/CNG): ").capitalize()
input_seller_type = input("Enter Seller Type (Dealer/Individual): ").capitalize()
input_transmission = input("Enter Transmission (Manual/Automatic): ").capitalize()

predicted_price = predict_new_car(
    year=input_year,
    present_price=input_present_price,
    kms_driven=input_kms_driven,
    owner=input_owner,
    fuel_type=input_fuel_type,
    seller_type=input_seller_type,
    transmission=input_transmission
)

print(f"\n=> The predicted selling price for this vehicle is: {predicted_price} Lakhs")

--- Enter Vehicle Details for Prediction ---
Enter the manufacturing year (e.g., 2017): 2016
Enter the present price in Lakhs (e.g., 9.85): 9.50
Enter kilometers driven (e.g., 6900): 7000
Enter number of previous owners (0, 1, 2...): 1
Enter Fuel Type (Petrol/Diesel/CNG): Petrol
Enter Seller Type (Dealer/Individual): Individual
Enter Transmission (Manual/Automatic): Manual

=> The predicted selling price for this vehicle is: 7.58 Lakhs
